# The classical view

Before you start: upload `daily_sales.csv` to Colab.


The data is made up: three years of daily sales for a fictional shop.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [2]:
df = pd.read_csv("daily_sales.csv")
df["date"] = pd.to_datetime(df["date"])

In [3]:
df = df.sort_values("date").reset_index(drop=True)
if df["date"].isna().any() or df["date"].duplicated().any():
    raise ValueError("The file must have one valid row per date.")
if not df["date"].equals(pd.Series(pd.date_range(df["date"].min(), df["date"].max(), freq="D"))):
    raise ValueError("The file must contain consecutive daily dates.")

In [4]:
train = df.iloc[:-60]
test = df.iloc[-60:]

## Stationarity
For weak stationarity, the mean and variance stay constant and dependence at a given lag does not change over time. ADF tests the narrower null hypothesis of a unit root. A small p-value is evidence against a unit root; a large p-value is inconclusive. Neither result checks every aspect of stationarity. Use training data for these diagnostics.

In [5]:
result = adfuller(train["sales"])
print("p-value of the raw training series:", result[1])

p-value of the raw training series: 0.8409804005020521


/tmp/ipykernel_15850/1353508954.py:1: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  result = adfuller(train["sales"])


## Differencing
Look at today minus yesterday. This can help with a changing level, but it does not guarantee stationarity or remove seasonal patterns.

In [6]:
train = train.copy()
train["change"] = train["sales"].diff()

fig = px.line(train, x="date", y="change", title="Day-to-day change in sales")
fig.show()

In [7]:
result = adfuller(train["change"].dropna())
print("p-value of the differenced series:", result[1])

p-value of the differenced series: 3.4171266504132884e-14


/tmp/ipykernel_15850/1274502072.py:1: FutureWarning:

adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.



## Autocorrelation
How strongly is the training series related to its values 1, 2, ... 14 days ago? A peak at lag 7 is consistent with weekly dependence. Inspect the output rather than assuming its shape.

In [8]:
values = acf(train["change"].dropna(), nlags=14)

fig = px.bar(x=list(range(15)), y=values, title="Autocorrelation of the daily change")
fig.update_layout(xaxis_title="days back", yaxis_title="correlation")
fig.show()

## Two baselines
Compare every model with a simple alternative on the same dates and forecast horizon. Here both baselines forecast all 60 days from one cutoff, without updating from test observations.

In [9]:
def mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))

actual = test["sales"].values

In [10]:
# Baseline 1: tomorrow = today. Repeat the last known value.
naive = np.repeat(train["sales"].iloc[-1], 60)

# Baseline 2: same day last week. Repeat the last 7 days.
last_week = train["sales"].iloc[-7:].values
seasonal_naive = np.tile(last_week, 9)[:60]

print("Naive MAE:         ", round(mae(actual, naive), 1))
print("Seasonal naive MAE:", round(mae(actual, seasonal_naive), 1))

Naive MAE:          31.8
Seasonal naive MAE: 24.3


## SARIMA
ARIMA with a weekly seasonal part. `order=(1,1,1)` is the regular part, `seasonal_order=(1,1,1,7)` is the same idea repeated every 7 days. We did not tune these numbers.

This specification models a weekly seasonal pattern, but does not explicitly model annual seasonality or holidays. SARIMAX can include calendar indicators as external regressors. This is one untuned specification, not a test of the entire ARIMA family.

In [11]:
model = SARIMAX(train["sales"].values, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7))
fitted = model.fit(disp=False)

sarima = fitted.forecast(60)
print("SARIMA MAE:", round(mae(actual, sarima), 1))

SARIMA MAE: 28.4


## SARIMAX with holidays
The same model, now with calendar information. We add one column per holiday: 1 on that holiday, 0 on every other day. These columns are external regressors, the X in SARIMAX. The holiday dates in the test period are known in advance, so using them does not use any test sales.

In [12]:
def holiday_indicators(dates):
    return pd.DataFrame({
        "thanksgiving": ((dates.dt.month == 11) & (dates.dt.weekday == 3) & dates.dt.day.between(22, 28)).astype(float),
        "christmas": ((dates.dt.month == 12) & (dates.dt.day == 25)).astype(float),
        "new_year": ((dates.dt.month == 1) & (dates.dt.day == 1)).astype(float),
    })

holidays_train = holiday_indicators(train["date"])
holidays_test = holiday_indicators(test["date"])

model_h = SARIMAX(train["sales"].values, exog=holidays_train.values,
                  order=(1, 1, 1), seasonal_order=(1, 1, 1, 7))
fitted_h = model_h.fit(disp=False)

sarimax_h = fitted_h.forecast(60, exog=holidays_test.values)
print("SARIMAX with holidays MAE:", round(mae(actual, sarimax_h), 1))

SARIMAX with holidays MAE: 22.0


## Choosing the orders
So far we used order (1, 1, 1) without checking it. A standard way to choose the number of lags is an information criterion:

 AIC (Akaike Information Criterion) and BIC (Bayesian Information Criterion)

Each one scores a fitted model: how well it fits the training data, with a penalty for extra complexity. Lower is better.

We fit each candidate order on the training data only. The test period is not used for this choice.

Another common way is a practice window: hold back the last part of the training data and compare forecast errors there.

In [13]:
import itertools
import warnings

results = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for p, q in itertools.product([0, 1, 2], [0, 1, 2]):
        m = SARIMAX(train["sales"].values, exog=holidays_train.values,
                    order=(p, 1, q), seasonal_order=(1, 1, 1, 7)).fit(disp=False)
        results.append({"p": p, "d": 1, "q": q, "AIC": round(m.aic, 1), "BIC": round(m.bic, 1)})

results = pd.DataFrame(results).sort_values("AIC").reset_index(drop=True)
results

,p,d,q,AIC,BIC
0,2,1,2,8141.7,8191.1
1,0,1,2,8182.5,8222.0
2,2,1,1,8182.5,8226.9
3,1,1,1,8182.7,8222.2
4,0,1,1,8182.8,8217.4
5,1,1,2,8183.2,8227.6
6,2,1,0,8332.5,8371.9
7,1,1,0,8445.8,8480.3
8,0,1,0,8668.4,8698.0


Lower is better. Now we take the best order, refit it, and score it once on the test period.

In [14]:
best_aic = results.sort_values("AIC").iloc[0]
best_bic = results.sort_values("BIC").iloc[0]
print("Best order by AIC:", (int(best_aic["p"]), 1, int(best_aic["q"])))
print("Best order by BIC:", (int(best_bic["p"]), 1, int(best_bic["q"])))

best_order = (int(best_aic["p"]), 1, int(best_aic["q"]))

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_tuned = SARIMAX(train["sales"].values, exog=holidays_train.values,
                          order=best_order, seasonal_order=(1, 1, 1, 7))
    fitted_tuned = model_tuned.fit(disp=False)

sarimax_tuned = fitted_tuned.forecast(60, exog=holidays_test.values)
print("Tuned SARIMAX with holidays, test MAE:", round(mae(actual, sarimax_tuned), 1))

Best order by AIC: (2, 1, 2)
Best order by BIC: (2, 1, 2)
Tuned SARIMAX with holidays, test MAE: 11.4


## Compare them on the test period

In [15]:
fig = go.Figure()
fig.add_scatter(x=test["date"], y=actual, name="actual", line=dict(color="black"))
fig.add_scatter(x=test["date"], y=naive, name="naive")
fig.add_scatter(x=test["date"], y=seasonal_naive, name="seasonal naive")
fig.add_scatter(x=test["date"], y=sarima, name="SARIMA")
fig.add_scatter(x=test["date"], y=sarimax_h, name="SARIMAX + holidays")
fig.add_scatter(x=test["date"], y=sarimax_tuned, name="SARIMAX + holidays, tuned")
fig.update_layout(title="Last 60 days: actual vs forecasts")
fig.show()

Inspect where the methods miss. Check holiday dates, trend changes, and other possible explanations. A pattern in the errors suggests something to investigate; it does not establish the cause.

If you change the model after looking at this test period, treat it as development feedback and reserve a new final test period.